In [1]:
import json
from collections import Counter 

import torch 
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

In [2]:
with open("/kaggle/input/datasets/shreyanshsharma049/q-and-a-dataset/train-v2.0 (1).json","r",encoding="utf8") as f:

    train_data = json.load(f)

print(train_data.keys())

dict_keys(['version', 'data'])


In [3]:
print("Number of Articles :",len(train_data["data"]))

Number of Articles : 442


In [4]:
print(train_data["data"][0].keys())

dict_keys(['title', 'paragraphs'])


In [5]:
paragraph = train_data["data"][0]["paragraphs"][0]

print(paragraph.keys())

dict_keys(['qas', 'context'])


In [6]:
print(paragraph["context"])

Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".


In [7]:
print(paragraph["qas"][0]["question"])

When did Beyonce start becoming popular?


In [8]:
print(paragraph["qas"][0]["answers"][0]["text"])

in the late 1990s


In [10]:
print(train_data["version"])

v2.0


In [11]:
samples = []

for article in train_data["data"]:

    for para in article["paragraphs"]:

        context = para["context"]

        for qa in para["qas"]:

            question = qa["question"]

            answer = qa["answers"]

            samples.append(

                (context, question, answer)

            )

In [12]:
print(len(samples))

130319


In [13]:
print(samples[0])

('Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'When did Beyonce start becoming popular?', [{'text': 'in the late 1990s', 'answer_start': 269}])


In [14]:
samples = []

for article in train_data["data"]:

    for para in article["paragraphs"]:

        context = para["context"]

        for qa in para["qas"]:

            question = qa["question"]

            # Skip unanswerable questions
            if len(qa["answers"]) == 0:
                continue

            answer = qa["answers"][0]["text"]

            samples.append(
                (context, question, answer)
            )

print("Total Samples :", len(samples))

Total Samples : 86821


In [15]:
print("Context:\n")
print(samples[0][0])

print("\nQuestion:\n")
print(samples[0][1])

print("\nAnswer:\n")
print(samples[0][2])

Context:

Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".

Question:

When did Beyonce start becoming popular?

Answer:

in the late 1990s


In [16]:
samples = samples[:5000]

print(len(samples))

5000


In [17]:
from collections import Counter

counter = Counter()

for context, question, answer in samples:

    counter.update(context.lower().split())

    counter.update(question.lower().split())

vocab = {

    "<PAD>":0,

    "<UNK>":1

}

for word in counter:

    vocab[word] = len(vocab)

print("Vocabulary Size :", len(vocab))

Vocabulary Size : 21478


In [18]:
def encode(text):

    return [

        vocab.get(word.lower(),1)

        for word in text.split()

    ]

In [19]:
print(samples[0][1])

print(encode(samples[0][1]))

When did Beyonce start becoming popular?
[91, 92, 93, 94, 95, 96]


In [20]:
MAX_CONTEXT = 200

MAX_QUESTION = 30

In [21]:
def pad_sequence(sequence, max_length):

    sequence = sequence[:max_length]

    while len(sequence) < max_length:

        sequence.append(0)

    return sequence

In [22]:
x = [1,2,3,4]

print(pad_sequence(x,10))

[1, 2, 3, 4, 0, 0, 0, 0, 0, 0]


In [23]:
class SquadDataset(Dataset):

    def __init__(self,samples):

        self.samples = samples

    def __len__(self):

        return len(self.samples)

    def __getitem__(self,index):

        context,question,answer = self.samples[index]

        context = pad_sequence(
            encode(context),
            MAX_CONTEXT
        )

        question = pad_sequence(
            encode(question),
            MAX_QUESTION
        )

        answer = torch.tensor(0)

        return (

            torch.tensor(context),

            torch.tensor(question),

            answer

        )

In [24]:
train_dataset = SquadDataset(samples)

print(len(train_dataset))

5000


In [25]:
context,question,answer = train_dataset[0]

print(context.shape)

print(question.shape)

print(answer)

torch.Size([200])
torch.Size([30])
tensor(0)


In [26]:
trainloader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=True

)

In [27]:
context,question,answer = next(iter(trainloader))

print(context.shape)

print(question.shape)

print(answer.shape)

torch.Size([32, 200])
torch.Size([32, 30])
torch.Size([32])


In [30]:
class SquadDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        context, question, answer = self.samples[index]

        # Context + Question
        text = context + " " + question

        text = encode(text)

        text = pad_sequence(text, 256)

        # Answer ko bhi encode karenge
        answer = encode(answer)

        if len(answer) == 0:
            answer = 1
        else:
            answer = answer[0]

        return (
            torch.tensor(text),
            torch.tensor(answer)
        )

In [31]:
train_dataset = SquadDataset(samples)

x, y = train_dataset[0]

print(x.shape)

print(y)

torch.Size([256])
tensor(22)


In [32]:
trainloader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=True

)

x, y = next(iter(trainloader))

print(x.shape)

print(y.shape)

torch.Size([32, 256])
torch.Size([32])


In [33]:
class QAModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(

            len(vocab),

            128

        )

        self.lstm = nn.LSTM(

            input_size=128,

            hidden_size=128,

            batch_first=True

        )

        self.fc = nn.Linear(

            128,

            len(vocab)

        )

    def forward(self,x):

        x = self.embedding(x)

        output,(hidden,cell)=self.lstm(x)

        output = self.fc(

            hidden[-1]

        )

        return output

In [34]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

model = QAModel().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(

    model.parameters(),

    lr=0.001

)

In [35]:
def train(model, loader, epochs):

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for x, y in loader:

            x = x.to(device)

            y = y.to(device)

            optimizer.zero_grad()

            output = model(x)

            loss = criterion(

                output,

                y

            )

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        print(

            f"Epoch {epoch+1} | Loss = {running_loss/len(loader):.4f}"

        )

In [36]:
train(

    model,

    trainloader,

    epochs=5

)

Epoch 1 | Loss = 8.5493
Epoch 2 | Loss = 7.1388
Epoch 3 | Loss = 6.9647
Epoch 4 | Loss = 6.9009
Epoch 5 | Loss = 6.8543


In [37]:
x, y = train_dataset[0]

model.eval()

with torch.no_grad():

    prediction = model(

        x.unsqueeze(0).to(device)

    )

predicted = prediction.argmax(1).item()

index_to_word = {

    value:key

    for key,value in vocab.items()

}

print("Predicted Answer :")

print(index_to_word.get(predicted,"UNK"))

Predicted Answer :
the
